# Day 10: End-to-End Orchestration, Observability & Verification

This notebook demonstrates a complete E2E run of a synthetic patient from raw JSON -> Bronze -> Tokenization -> Silver OMOP -> DQ -> Gold metric.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, expr, udf, when, date_add
from pyspark.sql.types import StringType
import hashlib
import datetime

spark = SparkSession.builder.appName("Day10_E2E_Walkthrough").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

### Step 1: Raw Payload Ingestion (Bronze)

In [ ]:
raw_hl7_data = [
    (1, "John Doe", "123-45-6789", datetime.date(1980, 5, 15), "V01", datetime.date(2023, 1, 1), datetime.date(2023, 1, 5)),
    (1, "John Doe", "123-45-6789", datetime.date(1980, 5, 15), "V02", datetime.date(2023, 1, 15), datetime.date(2023, 1, 16)) # Readmission
]

schema = ["patient_id", "name", "ssn", "dob", "visit_id", "visit_start", "visit_end"]
df_bronze = spark.createDataFrame(raw_hl7_data, schema) \
    .withColumn("ingestion_ts", current_timestamp())

print("1. Bronze Raw Ingestion:")
df_bronze.show()

### Step 2: Tokenization & De-Identification

In [ ]:
def fpe_encrypt(value):
    if not value: return None
    return hashlib.sha256(f"{value}_SECRET".encode()).hexdigest()[:11]

fpe_udf = udf(fpe_encrypt, StringType())

df_deid = df_bronze \
    .withColumn("ssn_masked", fpe_udf(col("ssn"))) \
    .drop("name", "ssn") # Drop PHI

print("2. De-Identified Payload:")
df_deid.show()

### Step 3: Data Quality Check

In [ ]:
df_dq = df_deid \
    .withColumn("is_valid", col("patient_id").isNotNull() & (col("dob") < current_timestamp().cast("date")))

df_valid = df_dq.filter(col("is_valid")).drop("is_valid")
print("3. Validated Records (Passed DQ):")
df_valid.show()

### Step 4: OMOP Silver Standardization

In [ ]:
df_omop_visit = df_valid.select(
    col("patient_id").alias("person_id"),
    col("visit_id").alias("visit_occurrence_id"),
    col("visit_start").alias("visit_start_date"),
    col("visit_end").alias("visit_end_date")
)

print("4. Silver OMOP Visit Occurrence:")
df_omop_visit.show()

### Step 5: Gold Metric (Readmissions)

In [ ]:
df_omop_visit.createOrReplaceTempView("omop_visit_occurrence")

gold_sql = """
WITH visits AS (
    SELECT 
        person_id,
        visit_occurrence_id,
        visit_start_date,
        visit_end_date,
        LAG(visit_end_date) OVER (PARTITION BY person_id ORDER BY visit_start_date) as prev_discharge_date
    FROM omop_visit_occurrence
)
SELECT
    person_id,
    COUNT(visit_occurrence_id) as total_visits,
    SUM(CASE WHEN prev_discharge_date IS NOT NULL AND datediff(visit_start_date, prev_discharge_date) <= 30 THEN 1 ELSE 0 END) as readmissions_within_30_days
FROM visits
GROUP BY person_id
"""

df_gold = spark.sql(gold_sql)
print("5. Gold Readmission Metrics (E2E Complete):")
df_gold.show()